## **Setup & Load Data**

In [ ]:
# install.packages("FNN")
# install.packages("ggrepel")
library(FNN)
library(lubridate)
library(tidyverse)
library(readr)
library(stringr)
library(bigrquery)
library(parallel)
library(ggrepel)

In [ ]:
EXPORT_BUCKET = "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055"
BILLING = "wb-silky-pepper-6055"
DATA_MOUNT = "/home/jupyter/workspace/raw/vwb-aou-datasets-controlled/v8"

In [ ]:
# Read query data directly from Cloud Storage into memory
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

In [ ]:
#TODO: need to revisit this - the auto-generated paths are only different by date, not cohort!
person_path <- "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055/bq_exports/person/20260509/person/person_*.csv"
person_df <- read_bq_export_from_workspace_bucket(person_path)

measurementOccurrence_path <- "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055/bq_exports/measurementOccurrence/20260510/measurementOccurrence/measurementOccurrence_*.csv"
measure_df <- read_bq_export_from_workspace_bucket(measurementOccurrence_path)

In [ ]:
# Read ancestry predictions
#TODO: why does this workspace have echo_v4_r2 prefix?
ancestry_path = paste0(DATA_MOUNT, "/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv")
ancestry_raw <- read_tsv(
  ancestry_path,
  show_col_types = FALSE,
  progress = FALSE)

## **Get Analytical Dataset**

For now, will only include:

+ TSH numeric
+ Participants with 3 or more TSH measures
+ Participants with ancestry PCs

Added mapping to "unit_concept_name" to SQL. Plan:

+ Condense/convert units according to the mapped unit_concept_name values
+ For those without a unit_concept_name (N=415,219) – assume units correct since TSH generally standard, but remove outliers

For now, could use value_as_number and remove outliers?

In [ ]:
# 1) Pull and clean HbA1c values
#TODO: add formal NA removal here
#TODO: re-add confirmation all UTC here?
head(measure_df)

In [ ]:
# Explore just values with unit
data_unit <- measure_df %>%
    dplyr::filter(!omop_unit_concept_name == "No matching concept")

table(data_unit$omop_unit_concept_name)
unique(data_unit$omop_unit_concept_name)

In [ ]:
# "micro-international unit per milliliter": median = 1.74, 0-929.2
# "international unit per milliliter": median = 1.52, 0.005-276.2
# "international unit per liter": median = 1.74, 0.008-74.212
# "milli-international unit per milliliter": median = 1.68, 0-1e07
# "micro-international unit per liter": median = 1.55, 0.006-263.6
# "No matching concept": median = 1.72, 0-100217

# Filter >=0 & <= 1000

# measure_df %>%
#   filter(omop_unit_concept_name %in% c(
#     "No matching concept"
#   )) %>%
#   summarise(
#     min = min(value_as_number, na.rm = TRUE),
#     median = median(value_as_number, na.rm = TRUE),
#     max = max(value_as_number, na.rm = TRUE),
#     n = n()
#   )

In [ ]:
# Standardize units, remove those that can't
    # Remove non-standard:
        # "no value", "unit", "times", "per milliliter", "microgram per deciliter",
        # "million per microliter", "percent"
    # Remove those with < 20 measurements (for prelim analysis)
        # "microunit per liter", "mU/L", "unit per milliliter", "milliunit per milliliter",
        # "uU/mL"
measure_df_mod <- measure_df %>%
    dplyr::filter(
        !omop_unit_concept_name %in% c(
            "no value", "unit", "times", "per milliliter", "microgram per deciliter",
            "million per microliter", "percent")) %>%
    dplyr::group_by(omop_unit_concept_name) %>%
    dplyr::filter(n() >= 20) %>%
    dplyr::ungroup()

In [ ]:
data <- measure_df_mod %>%
  transmute(
      tsh = as.numeric(value_as_number),
      person_id = as.character(format(person_id, scientific = FALSE, trim = TRUE)),
      datetime = ymd_hms(measurement_datetime, tz = "UTC")) %>%
  dplyr::filter(is.finite(tsh)) %>%
  dplyr::filter(tsh >= 0 & tsh < 1000)

dim(measure_df)
dim(measure_df_mod)
dim(data)

# Find IDs with >= 3 readings
ids_3plus <- data %>%
  count(person_id) %>%
  filter(n >= 3) %>%
  pull(person_id)

In [ ]:
# ----------------------------
# 2) Load ancestry PCs and keep only cohort participants
# ----------------------------
pc_cols <- paste0("pc", 1:16)
ancestry_pc <- ancestry_raw %>%
  transmute(
    research_id = as.character(format(research_id, scientific = FALSE, trim = TRUE)),
    # ancestry_pred_other = if ("ancestry_pred_other" %in% names(ancestry_raw)) ancestry_pred_other else NA_character_,
    pca_str = str_remove_all(pca_features, "\\[|\\]")
  ) %>%
  separate(
    pca_str,
    into = pc_cols,
    sep = ",\\s*",
    convert = TRUE,
    remove = TRUE
  )

In [ ]:
# Filter to match hba1c cohort above
ancestry_in_cohort <- ancestry_pc %>%
    dplyr::filter(research_id %in% ids_3plus)

# Filter to ensure above have PCs and 3+ measurements
data_anal <- data %>%
    dplyr::filter(person_id %in% ancestry_in_cohort$research_id) %>%
    dplyr::filter(person_id %in% ids_3plus)

ids_anal <- unique(data_anal$person_id)

# Summary of analytical dataset
cat("Number valid HbA1C measurements:", nrow(data), "\n")
cat("Participants with valid HbA1C:", n_distinct(data$person_id), "\n")
cat("Participants with >= 3 readings:", length(ids_3plus), "\n")
cat("Number valid HbA1C measurements after subset to >= 3 readings:", nrow(data_anal), "\n")
cat("Participants with valid HbA1C, >= 3 readings, & srWGS ancestry PCs:", nrow(ancestry_in_cohort), "\n")

In [ ]:
# Write out analysis data
write_tsv(data_anal, "data_analysis_tsh.txt")
system(paste0("gsutil cp data_analysis_tsh.txt ", EXPORT_BUCKET, "/"))

## **Get Matched Cohorts**

Alternate method for GenoSiS = using Euclidian distance on 16 ancestry PCs.

In [ ]:
# # Create the numeric matrix for fast distance calculation
# # Rows = people, Cols = PC1..PC16
# pc_matrix <- as.matrix(ancestry_in_cohort %>% column_to_rownames("research_id"))

In [ ]:
# k_neighbors <- 100
# # restrict_same_ancestry <- FALSE  # set TRUE to only search within same ancestry_pred_other
# #TODO: do I want to force matches to be within same ancestry group? - currently no

# knn_raw <- get.knn(pc_matrix, k=k_neighbors, algorithm="kd_tree")
# knn_cohort <- matrix(
#   rownames(pc_matrix)[knn_raw$nn.index],  # get id at index from input rownames
#   nrow = nrow(knn_raw$nn.index))  # same number rows as raw output
# rownames(knn_cohort) <- rownames(pc_matrix)  # add id that neighbors were found for

# # #TODO: do I want to save the actual distances too? - currently no

In [ ]:
# head(knn_cohort)
# dim(knn_cohort)

In [ ]:
# knn_cohort_t <- t(knn_cohort)
# head(knn_cohort_t)
# dim(knn_cohort_t)

In [ ]:
# # Save file
# write_tsv(as.data.frame(knn_cohort_t), "matched_cohorts_tsh.txt")
# system(paste0("gsutil cp matched_cohorts_tsh.txt ", EXPORT_BUCKET, "/"))

In [ ]:
# Reload file
system(paste0("gsutil cp ", EXPORT_BUCKET, "/matched_cohorts_tsh.txt ./"))
knn_cohort_t <- read_tsv("matched_cohorts_tsh.txt")
head(knn_cohort_t)
dim(knn_cohort_t)

## **Get Mean Shift for Every Individual**

Mean shift is calculated by using the shift in maxmimum density in the whole cohort compared to the matched cohort.

In [ ]:
# Global Density Peak
dim(data_anal)
full_d <- density(data_anal$tsh, na.rm = TRUE)
full_d_peak <- full_d$x[which.max(full_d$y)]

summary(data_anal$tsh)
print(full_d_peak)

In [ ]:
# For every person
    # Calculate their cohort's maximum A1c density
    # Calculate the shifted thresholds
get_shift <- function(id, cohort_ids) {
    cohort_ids <- unlist(cohort_ids, use.names = F)
    
    cohort <- data_anal[data_anal$person_id %in% cohort_ids, ]
    #TODO: could add average number measures per person in each cohort

    cohort_d <- density(cohort$tsh)  # checks on NA done & range already set
    cohort_d_peak <- cohort_d$x[which.max(cohort_d$y)]
    #TODO: read more about whether different density bandwidth should be used?

    shift <- full_d_peak - cohort_d_peak
    return(shift)
}

In [ ]:
#TO NOTE: these two rows must use the same ID list based on order
neighbor_list <- lapply(ids_anal, function(id) knn_cohort_t[, id])

In [ ]:
# detectCores()  # 4
# TO NOTE: this takes 20+ minutes as currently written

shift_values <- mclapply(seq_along(ids_anal), function(i) {
  id <- ids_anal[i]
  cohort_ids <- neighbor_list[[i]]
  get_shift(id, cohort_ids)
}, mc.cores = 4)

df <- data.frame(id = ids_anal, shift = unlist(shift_values))

In [ ]:
# Save file
write_tsv(df, "tsh_anal_shift.txt")
system(paste0("gsutil cp tsh_anal_shift.txt ", EXPORT_BUCKET, "/"))

In [ ]:
# Reload file
system(paste0("gsutil cp ", EXPORT_BUCKET, "/tsh_anal_shift.txt ./"))
df <- read.table("tsh_anal_shift.txt", header = TRUE)
dim(df)
head(df)